In [1]:
# pip install yfinance pandas_datareader matplotlib pandas numpy


import pandas as pd
import numpy as np
import yfinance as yf
from pandas_datareader import data as pdr
import datetime

START = "2024-01-01"
END   = datetime.date.today().strftime("%Y-%m-%d")
EQ_TICKER = "SPY"

eq_raw = yf.download(EQ_TICKER, start=START, end=END, progress=False, auto_adjust=True)
if isinstance(eq_raw.columns, pd.MultiIndex):
    eq_raw.columns = eq_raw.columns.get_level_values(0)
if "Close" in eq_raw.columns:
    equity = eq_raw[["Close"]].rename(columns={"Close":"equity"}).dropna()
elif "Adj Close" in eq_raw.columns:
    equity = eq_raw[["Adj Close"]].rename(columns={"Adj Close":"equity"}).dropna()
else:
    raise KeyError(f"No 'Close' or 'Adj Close' in columns: {list(eq_raw.columns)}")

y10 = pdr.DataReader("DGS10",  "fred", START, END).rename(columns={"DGS10":"y10"}).dropna()
y6m = pdr.DataReader("DGS6MO", "fred", START, END).rename(columns={"DGS6MO":"y6m"}).dropna()
usd = pdr.DataReader("DTWEXBGS", "fred", START, END).rename(columns={"DTWEXBGS":"usd"}).dropna()

eq_m  = equity.resample("M").last()
y10_m = y10.resample("M").last()
y6m_m = y6m.resample("M").last()
usd_m = usd.resample("M").last()

df = pd.concat([eq_m, y10_m, y6m_m, usd_m], axis=1, join="inner").dropna()
df.columns = ["equity","y10","y6m","usd"]

df["ret_equity_pct"] = df["equity"].pct_change() * 100.0
df["chg_y10_bp"]     = df["y10"].diff() * 100.0
df["chg_y6m_bp"]     = df["y6m"].diff() * 100.0
df["ret_usd_pct"]    = df["usd"].pct_change() * 100.0
df = df.dropna().reset_index().rename(columns={"index":"date"})

df["A_eq_dn_1pct"] = (df["ret_equity_pct"] <= -1.0).astype(int)   # ↓ ≥1% m/m
df["A_eq_up_1pct"] = (df["ret_equity_pct"] >= +1.0).astype(int)   # ↑ ≥1% m/m

df["A_y10_up"]  = (df["chg_y10_bp"] > 0.0).astype(int)
df["A_y10_dn"]  = (df["chg_y10_bp"] < 0.0).astype(int)
df["A_y6m_up"]  = (df["chg_y6m_bp"] > 0.0).astype(int)
df["A_y6m_dn"]  = (df["chg_y6m_bp"] < 0.0).astype(int)

df["B_usd_up"]  = (df["ret_usd_pct"] > 0.0).astype(int)
df["B_usd_dn"]  = (df["ret_usd_pct"] < 0.0).astype(int)

for col in ["B_usd_up","B_usd_dn","A_y10_up","A_y6m_up","A_y10_dn","A_y6m_dn", "A_eq_up_1pct","A_eq_dn_1pct"]:
    df[col + "_t1"] = df[col].shift(-1)

# ----- CN helper -----
def cn_stats(frame, A_col, B_col):
    sub = frame.dropna(subset=[A_col, B_col])
    A = sub[A_col] == 1
    B = sub[B_col] == 1
    n  = len(sub)
    nA = int(A.sum()); nB = int(B.sum())
    nAB = int((A & B).sum())
    pA = nA/n if n>0 else np.nan
    pB = nB/n if n>0 else np.nan
    pB_given_A = (nAB/nA) if nA>0 else np.nan
    pA_given_B = (nAB/nB) if nB>0 else np.nan
    eps = 1e-12
    lift = pB_given_A/(pB+eps) if pd.notna(pB_given_A) and pd.notna(pB) else np.nan
    nA_notB      = nA - nAB
    nNotA_B      = nB - nAB
    nNotA_notB   = n - (nAB + nA_notB + nNotA_B)
    odds_ratio   = ((nAB+eps)*(nNotA_notB+eps))/((nA_notB+eps)*(nNotA_B+eps))
    return pd.Series({
        "N": n, "N(A)": nA, "N(B)": nB, "N(A∩B)": nAB,
        "P(A)": pA, "P(B)": pB,
        "P(B|A)": pB_given_A, "P(A|B)": pA_given_B,
        "Lift": lift, "Odds Ratio": odds_ratio
    })

rows = []

def add_row(Acol, Bcol, label, horizon):
    s = cn_stats(df, Acol, Bcol).to_dict()
    s["Pair"] = label
    s["Horizon"] = horizon
    rows.append(s)

add_row("A_eq_up_1pct", "A_y10_up",        "Equities↑(≥+1% m/m) ⇒ 10y↑", "t")
add_row("A_eq_up_1pct", "A_y10_up_t1",     "Equities↑(≥+1% m/m) ⇒ 10y↑", "t+1")
add_row("A_eq_up_1pct", "A_y6m_up",        "Equities↑(≥+1% m/m) ⇒ 6m↑",  "t")
add_row("A_eq_up_1pct", "A_y6m_up_t1",     "Equities↑(≥+1% m/m) ⇒ 6m↑",  "t+1")

add_row("A_y10_dn",     "B_usd_dn",        "10y↓ ⇒ USD↓",                "t")
add_row("A_y10_dn",     "B_usd_dn_t1",     "10y↓ ⇒ USD↓",                "t+1")
add_row("A_y6m_dn",     "B_usd_dn",        "6m↓ ⇒ USD↓",                 "t")
add_row("A_y6m_dn",     "B_usd_dn_t1",     "6m↓ ⇒ USD↓",                 "t+1")

add_row("A_y10_up", "A_eq_up_1pct",      "10y↑ ⇒ Equities↑(≥+1% m/m)", "t")
add_row("A_y10_up", "A_eq_up_1pct_t1",   "10y↑ ⇒ Equities↑(≥+1% m/m)", "t+1")

add_row("A_y6m_up", "A_eq_up_1pct",      "6m↑ ⇒ Equities↑(≥+1% m/m)",  "t")
add_row("A_y6m_up", "A_eq_up_1pct_t1",   "6m↑ ⇒ Equities↑(≥+1% m/m)",  "t+1")

add_row("A_eq_dn_1pct", "B_usd_dn",    "Equities↓(≤−1% m/m) ⇒ USD↓", "t")
add_row("A_eq_dn_1pct", "B_usd_dn_t1", "Equities↓(≤−1% m/m) ⇒ USD↓", "t+1")

add_row("A_eq_dn_1pct", "A_y6m_dn",    "Equities↓(≤−1% m/m) ⇒ 6m↓",  "t")
add_row("A_eq_dn_1pct", "A_y6m_dn_t1", "Equities↓(≤−1% m/m) ⇒ 6m↓",  "t+1")

add_row("A_eq_dn_1pct", "B_usd_up",        "Equities↓(≤−1% m/m) ⇒ USD↑", "t")
add_row("A_eq_dn_1pct", "B_usd_up_t1",     "Equities↓(≤−1% m/m) ⇒ USD↑", "t+1")
add_row("A_y10_up",     "B_usd_up",        "10y↑ ⇒ USD↑",                "t")
add_row("A_y10_up",     "B_usd_up_t1",     "10y↑ ⇒ USD↑",                "t+1")
add_row("A_y6m_up",     "B_usd_up",        "6m↑ ⇒ USD↑",                 "t")
add_row("A_y6m_up",     "B_usd_up_t1",     "6m↑ ⇒ USD↑",                 "t+1")

add_row("B_usd_dn", "A_y10_dn", "USD↓ ⇒ 10y↓", "t")
add_row("B_usd_dn", "A_y10_dn_t1", "USD↓ ⇒ 10y↓", "t+1")
add_row("B_usd_dn", "A_y6m_dn", "USD↓ ⇒ 6m↓", "t")
add_row("B_usd_dn", "A_y6m_dn_t1", "USD↓ ⇒ 6m↓", "t+1")

add_row("B_usd_dn", "A_y10_up", "USD↓ ⇒ 10y↑", "t")
add_row("B_usd_dn", "A_y10_up_t1", "USD↓ ⇒ 10y↑", "t+1")
add_row("B_usd_dn", "A_y6m_up", "USD↓ ⇒ 6m↑", "t")
add_row("B_usd_dn", "A_y6m_up_t1", "USD↓ ⇒ 6m↑", "t+1")

add_row("B_usd_up", "A_y10_up",    "USD↑ ⇒ 10y↑", "t")
add_row("B_usd_up", "A_y10_up_t1", "USD↑ ⇒ 10y↑", "t+1")
add_row("B_usd_up", "A_y10_dn",    "USD↑ ⇒ 10y↓", "t")
add_row("B_usd_up", "A_y10_dn_t1", "USD↑ ⇒ 10y↓", "t+1")

add_row("B_usd_up", "A_y6m_up",    "USD↑ ⇒ 6m↑", "t")
add_row("B_usd_up", "A_y6m_up_t1", "USD↑ ⇒ 6m↑", "t+1")
add_row("B_usd_up", "A_y6m_dn",    "USD↑ ⇒ 6m↓", "t")
add_row("B_usd_up", "A_y6m_dn_t1", "USD↑ ⇒ 6m↓", "t+1")

out = pd.DataFrame(rows)[
    ["Pair","Horizon","N","N(A)","N(B)","N(A∩B)","P(A)","P(B)","P(B|A)","P(A|B)","Lift","Odds Ratio"]
].sort_values(["Pair","Horizon"]).reset_index(drop=True)

with pd.option_context('display.float_format', '{:.3f}'.format):
    print("\nConcurrent Necessity — Monthly 2024→present (vanilla thresholds)")
    print(out)



/Users/aidansinclair/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(



Concurrent Necessity — Monthly 2024→present (vanilla thresholds)
                          Pair Horizon      N   N(A)   N(B)  N(A∩B)  P(A)  \
0   10y↑ ⇒ Equities↑(≥+1% m/m)       t 19.000  6.000 13.000   3.000 0.316   
1   10y↑ ⇒ Equities↑(≥+1% m/m)     t+1 18.000  6.000 12.000   6.000 0.333   
2                  10y↑ ⇒ USD↑       t 19.000  6.000  7.000   5.000 0.316   
3                  10y↑ ⇒ USD↑     t+1 18.000  6.000  6.000   1.000 0.333   
4                  10y↓ ⇒ USD↓       t 19.000 12.000 12.000  10.000 0.632   
5                  10y↓ ⇒ USD↓     t+1 18.000 11.000 12.000   6.000 0.611   
6    6m↑ ⇒ Equities↑(≥+1% m/m)       t 19.000  7.000 13.000   5.000 0.368   
7    6m↑ ⇒ Equities↑(≥+1% m/m)     t+1 18.000  7.000 12.000   5.000 0.389   
8                   6m↑ ⇒ USD↑       t 19.000  7.000  7.000   4.000 0.368   
9                   6m↑ ⇒ USD↑     t+1 18.000  7.000  6.000   2.000 0.389   
10                  6m↓ ⇒ USD↓       t 19.000 12.000 12.000   9.000 0.632   
11        

/var/folders/43/9kgwqynj2f1b73147f7w9gj00000gn/T/ipykernel_29490/2070413061.py:28: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  eq_m  = equity.resample("M").last()
/var/folders/43/9kgwqynj2f1b73147f7w9gj00000gn/T/ipykernel_29490/2070413061.py:29: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  y10_m = y10.resample("M").last()
/var/folders/43/9kgwqynj2f1b73147f7w9gj00000gn/T/ipykernel_29490/2070413061.py:30: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  y6m_m = y6m.resample("M").last()
/var/folders/43/9kgwqynj2f1b73147f7w9gj00000gn/T/ipykernel_29490/2070413061.py:31: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  usd_m = usd.resample("M").last()
